# E-Commerce Marketplace Analysis (Olist): Revenue, Retention and Delivery

## Business case
You are the data analyst at **Olist**, an online marketplace in Brazil that connects sellers with customers. Leadership asks three questions:
1. **Where does our revenue come from?** (time, products, states, sellers)
2. **Why do customers rarely come back?** (retention)
3. **How much do late deliveries hurt customer satisfaction and repeat purchases?**

**Data:** Olist public dataset (9 related tables, about 100,000 orders, 2016-2018)
**Tools:** Python (pandas), SQL (SQLite), Power BI dashboard
**Approach:** load and check the data, join the tables into one analysis table, answer the questions with SQL, export clean files for Power BI, then write recommendations.

## Step 1: Unzip the data
Upload the Kaggle zip (usually named `archive.zip`) to the Colab sidebar, then run this cell.
*Why:* the dataset comes as one zip with 9 CSV files. This cell finds the zip and unpacks it.

In [1]:
import zipfile, glob, os
import pandas as pd
import sqlite3

zips = glob.glob('*.zip')
print('Zip files found:', zips)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('olist')
print(sorted(os.listdir('olist')))

Zip files found: ['archive (1).zip']
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Step 2: Load the tables
*Why:* each CSV is one table: customers, orders, order items, payments, reviews, products, sellers, and a category-name translation (Portuguese to English). We load them into pandas first.

In [2]:
d = 'olist/'
customers   = pd.read_csv(d + 'olist_customers_dataset.csv')
orders      = pd.read_csv(d + 'olist_orders_dataset.csv')
items       = pd.read_csv(d + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(d + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(d + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(d + 'olist_products_dataset.csv')
sellers     = pd.read_csv(d + 'olist_sellers_dataset.csv')
translation = pd.read_csv(d + 'product_category_name_translation.csv')

tables = {'customers': customers, 'orders': orders, 'order_items': items, 'payments': payments,
          'reviews': reviews, 'products': products, 'sellers': sellers, 'translation': translation}
for name, t in tables.items():
    print(f'{name:12s} rows={len(t):>7,}  columns={t.shape[1]}')

customers    rows= 99,441  columns=5
orders       rows= 99,441  columns=8
order_items  rows=112,650  columns=7
payments     rows=103,886  columns=5
reviews      rows= 99,224  columns=7
products     rows= 32,951  columns=9
sellers      rows=  3,095  columns=4
translation  rows=     71  columns=2


## Step 3: Put everything in a SQL database
Dates are converted to text in `YYYY-MM-DD HH:MM:SS` format, which SQLite can do date math on.

In [3]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c]).dt.strftime('%Y-%m-%d %H:%M:%S')

conn = sqlite3.connect('olist.db')
for name, t in tables.items():
    t.to_sql(name, conn, if_exists='replace', index=False)

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    conn.executescript(sql)
    conn.commit()

q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,customers
1,orders
2,order_items
3,payments
4,reviews
5,products
6,sellers
7,translation


## Step 4: Check the data before trusting it
Real data has traps. These checks explain *why* the joins below are written the way they are.

In [4]:
# Order status: we will analyze delivered orders only
q('''
SELECT order_status, COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM orders GROUP BY order_status ORDER BY orders DESC
''')

,order_status,orders,pct
0,delivered,96478,97.0
1,shipped,1107,1.1
2,canceled,625,0.6
3,unavailable,609,0.6
4,invoiced,314,0.3
5,processing,301,0.3
6,created,5,0.0
7,approved,2,0.0


In [5]:
# Trap 1: one order can have several payment rows, several items, even several reviews.
# Joining them directly would count revenue twice. So we summarize each table to ONE row per order first.
q('''
SELECT (SELECT COUNT(*) FROM (SELECT order_id FROM payments GROUP BY order_id HAVING COUNT(*) > 1)) AS orders_with_multiple_payments,
       (SELECT COUNT(*) FROM (SELECT order_id FROM order_items GROUP BY order_id HAVING COUNT(*) > 1)) AS orders_with_multiple_items,
       (SELECT COUNT(*) FROM (SELECT order_id FROM reviews GROUP BY order_id HAVING COUNT(*) > 1)) AS orders_with_multiple_reviews
''')

,orders_with_multiple_payments,orders_with_multiple_items,orders_with_multiple_reviews
0,2961,9803,547


In [6]:
# Trap 2: customer_id is new for every order. The real person is customer_unique_id.
q('''
SELECT COUNT(DISTINCT customer_id) AS customer_ids, COUNT(DISTINCT customer_unique_id) AS real_customers
FROM customers
''')

,customer_ids,real_customers
0,99441,96096


## Step 5: Build one clean analysis table (`fact_orders`)
One row per delivered order, with revenue, review score, delivery time, and whether it was late.
* Each source table is first summarized to one row per order (CTEs), so nothing is double counted.
* **Revenue** = sum of item prices (freight is kept separately).
* **Late** = delivered after the estimated delivery date.
* `julianday()` turns a date into a number of days, so subtracting two dates gives days.

In [7]:
run('''
DROP TABLE IF EXISTS fact_orders;
CREATE TABLE fact_orders AS
WITH order_value AS (
    SELECT order_id, COUNT(*) AS items, SUM(price) AS revenue, SUM(freight_value) AS freight
    FROM order_items GROUP BY order_id
),
order_review AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM reviews GROUP BY order_id
)
SELECT o.order_id,
       c.customer_unique_id,
       c.customer_state,
       o.order_purchase_timestamp AS purchase_ts,
       substr(o.order_purchase_timestamp, 1, 7) AS purchase_month,
       ov.items, ov.revenue, ov.freight,
       r.review_score,
       ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp), 1) AS delivery_days,
       ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date), 1) AS delay_days,
       CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date)
            THEN 1 ELSE 0 END AS is_late
FROM orders o
JOIN customers c     ON o.customer_id = c.customer_id
JOIN order_value ov  ON o.order_id = ov.order_id
LEFT JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;
''')
q('SELECT COUNT(*) AS delivered_orders FROM fact_orders')

,delivered_orders
0,96470


## Question 1: Headline KPIs
*What does the business look like at a glance?* Orders, real customers, revenue, average order value, average review score, and the share of late deliveries.

In [8]:
q('''
SELECT COUNT(*)                              AS orders,
       COUNT(DISTINCT customer_unique_id)    AS customers,
       ROUND(SUM(revenue), 0)                AS revenue,
       ROUND(AVG(revenue), 1)                AS avg_order_value,
       ROUND(AVG(review_score), 2)           AS avg_review,
       ROUND(100.0 * AVG(is_late), 1)        AS late_pct
FROM fact_orders
''')

,orders,customers,revenue,avg_order_value,avg_review,late_pct
0,96470,93350,13220249.0,137.0,4.16,8.1


## Question 2: How is revenue growing month by month?
`LAG()` is a **window function**: it looks at the previous row, so we can calculate month-over-month growth without a self-join.
We start from 2017 because 2016 has very few orders.

In [9]:
q('''
WITH monthly AS (
    SELECT purchase_month, COUNT(*) AS orders, SUM(revenue) AS revenue
    FROM fact_orders
    WHERE purchase_month >= '2017-01'
    GROUP BY purchase_month
)
SELECT purchase_month, orders, ROUND(revenue, 0) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY purchase_month))
             / LAG(revenue) OVER (ORDER BY purchase_month), 1) AS mom_growth_pct
FROM monthly
ORDER BY purchase_month
''')

,purchase_month,orders,revenue,mom_growth_pct
0,2017-01,750,111798.0,NaN
1,2017-02,1653,234223.0,109.5
2,2017-03,2546,359199.0,53.4
3,2017-04,2303,340670.0,-5.2
4,2017-05,3545,489159.0,43.6
5,2017-06,3135,421923.0,-13.7
6,2017-07,3872,481605.0,14.1
7,2017-08,4193,554700.0,15.2
8,2017-09,4150,607400.0,9.5
9,2017-10,4478,648248.0,6.7


## Question 3: Which product categories earn the most?
Here we join items to products and the translation table to get English category names, then use `SUM(...) OVER ()` to get each category's share of total revenue.

In [10]:
q('''
SELECT COALESCE(t.product_category_name_english, 'unknown') AS category,
       COUNT(DISTINCT i.order_id)                            AS orders,
       ROUND(SUM(i.price), 0)                                AS revenue,
       ROUND(100.0 * SUM(i.price) / SUM(SUM(i.price)) OVER (), 1) AS share_pct
FROM order_items i
JOIN fact_orders f          ON i.order_id = f.order_id
LEFT JOIN products p        ON i.product_id = p.product_id
LEFT JOIN translation t     ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY revenue DESC
LIMIT 10
''')

,category,orders,revenue,share_pct
0,health_beauty,8647,1233132.0,9.3
1,watches_gifts,5493,1165899.0,8.8
2,bed_bath_table,9272,1023435.0,7.7
3,sports_leisure,7529,954674.0,7.2
4,computers_accessories,6529,888614.0,6.7
5,furniture_decor,6307,711928.0,5.4
6,housewares,5743,615629.0,4.7
7,cool_stuff,3559,610204.0,4.6
8,auto,3809,578849.0,4.4
9,toys,3803,471097.0,3.6


## Question 4: Which states drive revenue?

In [11]:
q('''
SELECT customer_state,
       COUNT(*)                                            AS orders,
       ROUND(SUM(revenue), 0)                              AS revenue,
       ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1) AS share_pct
FROM fact_orders
GROUP BY customer_state
ORDER BY revenue DESC
LIMIT 10
''')

,customer_state,orders,revenue,share_pct
0,SP,40494,5066563.0,38.3
1,RJ,12350,1759651.0,13.3
2,MG,11354,1552482.0,11.7
3,RS,5344,728718.0,5.5
4,PR,4923,666064.0,5.0
5,SC,3546,507012.0,3.8
6,BA,3256,493584.0,3.7
7,DF,2080,296498.0,2.2
8,GO,1957,282837.0,2.1
9,ES,1995,268643.0,2.0


## Question 5: Do customers come back? (retention)
Count orders per *real* customer (`customer_unique_id`), then see what share bought more than once.
If this number is very low, it is a major business problem: the company keeps paying to acquire customers who never return.

In [12]:
q('''
WITH per_customer AS (
    SELECT customer_unique_id, COUNT(*) AS orders
    FROM fact_orders GROUP BY customer_unique_id
)
SELECT COUNT(*)                                                    AS customers,
       SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END)                 AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS repeat_rate_pct
FROM per_customer
''')

,customers,repeat_customers,repeat_rate_pct
0,93350,2801,3.0


## Question 6: Do late deliveries hurt reviews?
Compare on-time and late orders by average review score and by the share of bad reviews (1 or 2 stars).

In [13]:
q('''
SELECT CASE WHEN is_late = 1 THEN 'Late' ELSE 'On time' END AS delivery,
       COUNT(*)                                                    AS orders,
       ROUND(AVG(review_score), 2)                                 AS avg_review,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 1) AS pct_bad_reviews
FROM fact_orders
WHERE review_score IS NOT NULL
GROUP BY is_late
''')

,delivery,orders,avg_review,pct_bad_reviews
0,On time,88163,4.29,9.2
1,Late,7661,2.57,54.0


In [14]:
# How bad does it get as delays grow?
q('''
SELECT CASE WHEN delay_days <= 0 THEN '1) On time or early'
            WHEN delay_days <= 3 THEN '2) 1-3 days late'
            WHEN delay_days <= 7 THEN '3) 4-7 days late'
            ELSE                      '4) 8+ days late' END AS delay_band,
       COUNT(*)                     AS orders,
       ROUND(AVG(review_score), 2)  AS avg_review
FROM fact_orders
WHERE review_score IS NOT NULL
GROUP BY delay_band
ORDER BY delay_band
''')

,delay_band,orders,avg_review
0,1) On time or early,88224,4.29
1,2) 1-3 days late,2591,3.75
2,3) 4-7 days late,1787,2.30
3,4) 8+ days late,3222,1.73


## Question 7: Which states have the worst delivery performance?
`HAVING` keeps only states with enough orders (300+), so small states don't distort the ranking.

In [15]:
q('''
SELECT customer_state,
       COUNT(*)                        AS orders,
       ROUND(100.0 * AVG(is_late), 1)  AS late_pct,
       ROUND(AVG(delivery_days), 1)    AS avg_delivery_days,
       ROUND(AVG(review_score), 2)     AS avg_review
FROM fact_orders
GROUP BY customer_state
HAVING COUNT(*) >= 300
ORDER BY late_pct DESC
LIMIT 8
''')

,customer_state,orders,late_pct,avg_delivery_days,avg_review
0,AL,397,23.9,24.5,3.85
1,MA,717,19.7,21.6,3.83
2,PI,476,16.0,19.5,3.99
3,CE,1279,15.3,21.3,3.94
4,SE,335,15.2,21.5,3.91
5,BA,3256,14.0,19.3,3.93
6,RJ,12350,13.5,15.3,3.97
7,PA,946,12.4,23.8,3.91


## Question 8: How much damage do late deliveries do?
Two numbers a manager can act on: how much revenue sits in late orders, and what share of **all bad reviews** come from late orders.

In [16]:
q('''
SELECT ROUND(SUM(CASE WHEN is_late = 1 THEN revenue END), 0)                              AS late_order_revenue,
       ROUND(100.0 * SUM(CASE WHEN is_late = 1 THEN revenue END) / SUM(revenue), 1)       AS late_revenue_pct,
       ROUND(100.0 * SUM(CASE WHEN is_late = 1 AND review_score <= 2 THEN 1 ELSE 0 END)
                   / SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 1)               AS pct_of_bad_reviews_from_late
FROM fact_orders
''')

,late_order_revenue,late_revenue_pct,pct_of_bad_reviews_from_late
0,1158921.0,8.8,33.8


## Question 9 (key insight): Does a late first order reduce repeat purchases?
* `ROW_NUMBER() OVER (PARTITION BY customer ORDER BY date)` numbers each customer's orders in time order, so `rn = 1` is the first order.
* `COUNT(*) OVER (PARTITION BY customer)` gives each customer's total orders.
* We then compare the repeat rate of customers whose first order was late vs on time. This connects delivery (Question 6) to retention (Question 5).

In [17]:
q('''
WITH ranked AS (
    SELECT customer_unique_id, is_late,
           ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY purchase_ts) AS rn,
           COUNT(*)     OVER (PARTITION BY customer_unique_id)                      AS total_orders
    FROM fact_orders
)
SELECT CASE WHEN is_late = 1 THEN 'First order late' ELSE 'First order on time' END AS first_order,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END), 2) AS repeat_rate_pct
FROM ranked
WHERE rn = 1
GROUP BY is_late
''')

,first_order,customers,repeat_rate_pct
0,First order on time,85746,3.04
1,First order late,7604,2.51


## Question 10: Are we dependent on a few sellers?
Top 10 sellers by revenue, with their share of total revenue, average review, and late-delivery rate.

In [18]:
q('''
SELECT i.seller_id,
       COUNT(DISTINCT i.order_id)                                    AS orders,
       ROUND(SUM(i.price), 0)                                        AS revenue,
       ROUND(100.0 * SUM(i.price) / SUM(SUM(i.price)) OVER (), 2)    AS share_pct,
       ROUND(AVG(f.review_score), 2)                                 AS avg_review,
       ROUND(100.0 * AVG(f.is_late), 1)                              AS late_pct
FROM order_items i
JOIN fact_orders f ON i.order_id = f.order_id
GROUP BY i.seller_id
ORDER BY revenue DESC
LIMIT 10
''')

,seller_id,orders,revenue,share_pct,avg_review,late_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226988.0,1.72,4.14,11.6
1,53243585a1d6dc2643021fd1853d8905,348,217940.0,1.65,4.13,4.0
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.0,1.49,3.83,11.0
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.0,1.44,4.37,10.2
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.0,1.41,3.35,9.6
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.0,1.26,4.36,5.6
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159817.0,1.21,4.07,7.3
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139659.0,1.06,4.27,5.9
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138209.0,1.05,3.87,9.2
9,955fee9216a65b617aa5c0531780ce60,1261,131837.0,1.00,4.09,8.1


## Step 6: Export clean files for the Power BI dashboard
Two files: one row per order (`fact_orders.csv`) and one row per item with its category (`fact_items.csv`).

In [19]:
fact_orders = q('SELECT * FROM fact_orders')
fact_items = q('''
SELECT i.order_id, i.seller_id,
       COALESCE(t.product_category_name_english, 'unknown') AS category,
       i.price, i.freight_value,
       f.customer_state, f.purchase_month, f.is_late, f.review_score
FROM order_items i
JOIN fact_orders f      ON i.order_id = f.order_id
LEFT JOIN products p    ON i.product_id = p.product_id
LEFT JOIN translation t ON p.product_category_name = t.product_category_name
''')
fact_orders.to_csv('fact_orders.csv', index=False)
fact_items.to_csv('fact_items.csv', index=False)
print('fact_orders:', fact_orders.shape, '| fact_items:', fact_items.shape)

try:
    from google.colab import files
    files.download('fact_orders.csv')
    files.download('fact_items.csv')
except ImportError:
    pass

fact_orders: (96470, 12) | fact_items: (110189, 9)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Findings
1. Scale: 96,470 delivered orders from 93,350 customers generated 13.2M in revenue (Brazilian reais), with an average order value of 137.
2. Satisfaction: the average review is 4.16, and 8.1% of orders arrive after the estimated date.
3. Retention is the biggest problem: only 3.0% of customers ever placed a second order, so 97% bought once.
4. Late delivery sharply lowers reviews, and the damage grows with the delay: 4.29 (on time), 3.75 (1-3 days late), 2.30 (4-7 days late), 1.73 (8+ days late).
5. Late orders are 8.1% of orders but account for 33.8% of all bad (1-2 star) reviews, about four times their share. They also hold 1.16M (8.8%) of revenue.
6. Customers whose first order was late repeat at 2.51%, versus 3.04% for on-time first orders. Delivery matters, but it does not explain the low retention on its own.

Recommendations
1. Prioritize severe delays (4+ days late): about 5,000 orders with an average review below 2.0.
2. Build retention with separate levers (post-purchase emails, second-order vouchers) and test them, since even on-time customers rarely return.
3. Give proactive updates on delayed orders and review the delivery estimates, then check whether reviews improve.
4. Track repeat rate and late-delivery rate every month as core KPIs.

Limitations: data covers 2016-2018 from one marketplace, shows associations (not proof of cause), and repeat purchases are only measured within the dataset window.
